In [39]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage,SystemMessage,AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser,JsonOutputParser
from pydantic import BaseModel,Field
from dotenv import load_dotenv
import os

API key validation

In [44]:
from google.genai.errors import APIError
llm_gemini=ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")
try:
    response = llm_gemini.invoke("Hello")
    print(response.content)
except Exception as e:
    print("Raw Error:", e)

[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'El4KXAERTTIPelgIj4QFa2QE+YHbXLkBusGrvCPfxfUcsodGWgHzTgbv0y+GfXUQA1cPVZbGH/sRGIVb3mkqO6ScJBxDgoRvKA9eS1hSLHMWaPOnZgI0IwsoYUCUTBay'}}]


Envirnoment Varible Checking

In [22]:
if os.environ.get("GOOGLE_API_KEY"):
    print("api key is found that is gemini")
else:
    raise ValueError("GOOGLE_API_KEY environment variable not set")

api key is found that is gemini


String Output Parsing


In [31]:
from langchain_core.output_parsers import format_instructions
llm_gemini=ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")
prompt_templete=ChatPromptTemplate.from_messages([
    ("system","you are a good teacher"),
    ("user","teach me about {topic}")
])
parser=StrOutputParser()
user_input=input("enter a topic")
ready_prompt=prompt_templete.invoke({"topic": user_input})
response=llm_gemini.invoke(ready_prompt)
parser.invoke(response)


'Welcome! I am so excited to teach you about AI (Artificial Intelligence). Think of me as your guide. We will take this step-by-step, with no confusing jargon unless we define it together. \n\nTo start, let’s answer the most important question: **What actually is AI?**\n\n### 1. The Simple Definition\nAt its core, **Artificial Intelligence is the ability of a computer or machine to mimic human intelligence.** \n\nWhile a normal computer just follows strict, pre-written instructions (like a calculator), an **AI system can learn, reason, solve problems, perceive its environment, and even understand language.**\n\n### 2. A Quick Analogy\n*   **A Normal Computer:** Imagine a robot programmed to bake a cake. If you tell it, "Mix flour, sugar, and eggs," it will do that. But if you hide the eggs, the normal computer will crash or stop because it doesn\'t know what to do outside of its exact instructions.\n*   **An AI Computer:** This robot would look around, realize you are out of eggs, figu

JSON Output Parsing

In [ ]:
llm_gemini=ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")
jsonparser=JsonOutputParser()
prompt_templete=ChatPromptTemplate.from_messages([
    ("system","you are a good teacher\n. {format_instructions}"),
    ("user","teach me about {topic}")
]).partial(format_instructions=jsonparser.get_format_instructions())# requried in json parsing
user_input = input("Enter a topic: ")
ready_prompt = prompt_templete.invoke({"topic": user_input})
response=llm_gemini.invoke(ready_prompt)
jsonparser.invoke(response)


{'topic': 'Artificial Intelligence (AI)',
 'lesson_level': 'Beginner',
 'introduction': 'Welcome! Artificial Intelligence, or AI, sounds like science fiction, but it is actually a part of your everyday life. Think of AI as teaching machines—like computers and smartphones—to think, learn, and solve problems similar to how humans do.',
 'core_concepts': [{'title': '1. What is AI?',
   'description': 'At its core, AI is a branch of computer science focused on building smart machines capable of performing tasks that typically require human intelligence, such as recognizing speech, making decisions, and translating languages.'},
  {'title': '2. Machine Learning (ML) vs. Deep Learning (DL)',
   'description': "Machine Learning is a subset of AI where computers learn from data without being explicitly programmed for every single rule. Deep Learning is an advanced subset of ML inspired by the human brain, using 'neural networks' to process complex patterns (like recognizing faces in photos)."}

Pydantic Object Parsing

In [50]:
from langchain_core.output_parsers import PydanticOutputParser
from traitlets.utils import descriptions
llm_gemini=ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")
class Teacher(BaseModel):
    one_sentence:str= Field(descriptions="answer in one sentence")
    things:str=Field(descriptions="Answer for the queries")
pydantic_parser=PydanticOutputParser(pydantic_object=Teacher)
prompt_templete=ChatPromptTemplate.from_messages([
    ("system","you are a good teacher\n. {format_instructions}"),
    ("user","teach me about {topic}")
]).partial(format_instructions=pydantic_parser.get_format_instructions())# requried in pydantic parsing
user_input = input("Enter a topic: ")
ready_prompt = prompt_templete.invoke({"topic": user_input})
response=llm_gemini.invoke(ready_prompt)
pydantic_output=pydantic_parser.invoke(response)
pydantic_output.things
pydantic_output.one_sentence

C:\Users\Komalkumar M A\AppData\Local\Temp\ipykernel_5644\1994480135.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'descriptions'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  one_sentence:str= Field(descriptions="answer in one sentence")
C:\Users\Komalkumar M A\AppData\Local\Temp\ipykernel_5644\1994480135.py:6: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'descriptions'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  things:str=Field(descriptions="Answer for the queries")


'Artificial Intelligence is the simulation of human intelligence by machines, enabling them to learn, reason, and solve complex problems.'